## 1. LLM 최적화 전략별 종합 트레이드오프 (Trade-off)

실무에서 프로젝트의 요구 사항에 따라 어떤 기법을 선택해야 하는지 결정하기 위한 핵심 기준은 다음과 같습니다.

| 비교 항목 | 프롬프트 엔지니어링 (Prompt Eng.) | RAG (검색 증강 생성) | PEFT (LoRA 등) | 풀 파인튜닝 (Full FT) | RLHF / DPO |
|:---|:---|:---|:---|:---|:---|
| **필요 데이터량** | 매우 소량 (수 개) | 사내 지식 문서 데이터 | 소~중형 (수백~수만 개) | 대량 (수만~수백만 개) | 선호도 데이터 (수천 개) |
| **GPU 학습 리소스** | 필요 없음 (0) | 필요 없음 (0) | 소형 GPU 1대로 가능 | 수십~수백 대 GPU 필요 | 중~대형 GPU 클러스터 필요 |
| **초기 개발 난이도**| 매우 낮음 | 보통 (인덱싱 필요) | 보통 (코드 구현 필요) | 높음 (인프라 필요) | 매우 높음 (정렬 피드백) |
| **지식 갱신 주기** | 즉시 (프롬프트 변경) | 실시간 (DB 업데이트) | 재학습 주기 필요 | 매우 긴 재학습 주기 | 매우 긴 재학습 주기 |
| **주요 목적** | 빠른 검증, 일반 질의 | 사내 문서 기반 사실 답변 | 스타일, 어조, 도메인 적응 | 기초 모델 개발, 도메인 특화 | 인간 윤리 및 선호 정렬 |

In [19]:
import time
import torch
import pandas as pd 
from transformers import AutoTokenizer, AutoModelForCausalLM
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

### 2. 평가 시나리오: 회사 FAQ답변 평가

사내 환불 규정 및 요금 정책에 대한 사용자 질문에 대응하는 세 가지 아키텍처 모델의 출력을 준비하고, 이를 Ground Truth(정답 가이드)와 비교하여 정량 평가해 봅니다.

- 시나리오 구성:
    1. Ground Truth: 사내 가이드 상의 모범 답안
    2. Base Model (gpt2): 사내 정보가 학습되지 않아 일반 웹 문서 스타일로 답변하는 기본 모델 출력
    3. RAG Model: 관련 내부 문서를 프롬프트 컨텍스트에 동봉하여 사실 기반으로 답변하는 아키텍쳐 출력
    4. Fine-tuned Model: 사내 FAQ 데이터를 학습하여 간결하게 양식에 맞춰 답변하는 어댑터 모델 출력

In [20]:
# 평가용 데이터 정의
eval_faq_data = [
    {
        "question": "What is the company refund policy?",
        "ground_truth": "Customers can request a full refund within 14 days of purchase if the service is unused.",
        "base_model_output": "Refund policy varies depending on the country. Please check our global website for more details or contact sales.",
        "rag_model_output": "According to our official guide, you can get a full refund within 14 days of purchase as long as the service remains unused.",
        "finetuned_model_output": "Refund policy: Full refund within 14 days of purchase for unused services."
    },
    {
        "question": "Is there an extra charge for weekend support?",
        "ground_truth": "No, standard weekend email support is included in all premium plans without extra charges.",
        "base_model_output": "Support services are usually provided on weekdays. Weekend calls might be billed per hour.",
        "rag_model_output": "Standard weekend email support is included in premium plans with no extra charges.",
        "finetuned_model_output": "Weekend support charges: standard email support is free of charge on premium plans."
    }
 ]
df_eval = pd.DataFrame(eval_faq_data)
print("Evaluation dataset initialized!")

Evaluation dataset initialized!


In [29]:
import re
# 정규화(전처리) 함수
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\'\s]+', '', text)
    return text.strip()

# 1. Exact Match ( 정확 매칭 여부 )
def compute_exact_match(prediction, ground_truth):
    return normalize_text(prediction) == normalize_text(ground_truth)

# 2. Token-level F1 Score
def compute_token_f1(prediction, ground_truth):
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(ground_truth).split()

    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return 1.0 if len(pred_tokens) == len(truth_tokens) else 0.0
    
    common = set(pred_tokens) & set(truth_tokens)
    num_same = len(common)
    if num_same == 0: return 0.0

    # 정밀도 (암환자)
    precision = num_same / len(pred_tokens)
    # 재현율 (스팸메일_)
    recall = num_same / len(truth_tokens)
    f1 = 2*(precision * recall) / (precision + recall)
    return f1

# 3. ROUGE-1 Overlap Precision / recall / F1 => 요약 테스트(긴 문장)
def compute_rouge1(prediction, ground_truth):
    # 토큰화
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(ground_truth).split()
    # ROUGE-1 unigram(단일 단어)기반 비교
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return 0.0, 0.0, 0.0
    
    # 공통토큰 추출
    common = set(pred_tokens) & set(truth_tokens)
    overlap_count = len(common)

    precision = overlap_count / len(pred_tokens)
    recall = overlap_count / len(truth_tokens)
    f1 = 2*(precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

### 4. 모델별 평가 지표 계산 및 결과 집계
준비된 세 가지 모델 아키텍처 출력을 정답 FAQ 가이드와 비교하여 EM, F1, ROUGE-1 지표의 평균값을 도출해 봅니다.

In [30]:
models = ['base_model', 'rag_model', 'finetuned_model']
summary_matrics = []
for model_name in models:
    col_name = f'{model_name}_output'
    em_sum = 0
    f1_sum = 0
    r1_p_sum = 0
    r1_r_sum = 0
    r1_f_sum = 0
    n = len(df_eval)
    for idx, row in df_eval.iterrows():
        pred = row[col_name]
        truth = row['ground_truth']

        em_sum += compute_exact_match(pred,truth)
        f1_sum += compute_token_f1(pred, truth)
        p, r, f = compute_rouge1(pred, truth)
        r1_p_sum += p
        r1_r_sum += r
        r1_f_sum += f
    summary_matrics.append({
        'Architecture Model': model_name.upper(),
        'Exact Match (EM)': f'{em_sum / n:.2%}',
        'Token F1 Score': f'{f1_sum / n:.2%}',
        'ROUGE F1 Precision': f'{r1_p_sum / n:.2%}',
        'ROUGE F1 Recall': f'{r1_r_sum / n:.2%}',
        'ROUGE F1 Score': f'{r1_f_sum / n:.2%}',
    })
df_summary = pd.DataFrame(summary_matrics)
print(df_summary.to_string(index=False))


Architecture Model Exact Match (EM) Token F1 Score ROUGE F1 Precision ROUGE F1 Recall ROUGE F1 Score
        BASE_MODEL            0.00%         13.03%             12.70%          13.39%         13.03%
         RAG_MODEL            0.00%         75.21%             72.24%          80.36%         75.21%
   FINETUNED_MODEL            0.00%         58.20%             64.10%          53.57%         58.20%
